In [ ]:
import geopandas as gpd

mapping = gpd.read_file(r'F:\Workspace\世界国家_new.shp')

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from shapely.geometry import mapping
import rasterio
from rasterio.enums import Resampling

In [ ]:
# 打开包含风速、温度和太阳辐射的 NetCDF 文件
# 文件名: "6windtempsolar201920.nc"
slope = xr.open_dataset(r"F:\Workspace\DATA_DEM\Slope_map_1km.tif")["band_data"][0, :, :] #作为投影参考
data = xr.open_dataset(r"E:\Workspace\2008.nc")

# 提取 100 米高度的东向风速 (u 分量) 和北向风速 (v 分量)
# 这里使用 8760 索引开始，表示从第二年的数据开始（假设每小时记录一次）
i=0
u100 = data["u100"][0:8760, :, :]
v100 = data["v100"][0:8760, :, :]
u100 = u100.rio.write_crs(slope.rio.crs) #重新进行投影
v100 = v100.rio.write_crs(slope.rio.crs) #重新进行投影

In [ ]:
slope_025d = xr.open_dataset(r"F:\Workspace\DATA_DEM\Slope_map_025d.tif")["band_data"][0, :, :]
slope_1km = xr.open_dataset(r"F:\Workspace\DATA_DEM\Slope_map_1km.tif")["band_data"][0, :, :]
lucc_025d = xr.open_dataset(r"F:\Workspace\DATA_Landuse\World_landuse_025d.tif")["band_data"][0, :, :]
lucc_1km = xr.open_dataset(r"F:\Workspace\DATA_Landuse\World_landuse_1km.tif")["band_data"][0, :, :]
elevation_025d = xr.open_dataset(r"F:\Workspace\DATA_DEM\merged_global_map_025d.tif")["band_data"][0, :, :]
elevation_1km = xr.open_dataset(r"F:\Workspace\DATA_DEM\merged_global_map.tif")["band_data"][0, :, :]
protected_025d = xr.open_dataset(r"F:\Workspace\DATA_Protected\Protected_025d.tif")["band_data"][0, :, :].rio.reproject_match(u100, resampling=Resampling.mode)
protected_1km = xr.open_dataset(r"F:\Workspace\DATA_Protected\Protected_1km.tif")["band_data"][0, :, :]
classification_df = pd.read_csv("land_use_classification.csv")
lucc_Wind_weights = dict(zip(classification_df['Flag Value'], classification_df['Onshore wind-Base']))
lucc_Solar_weights = dict(zip(classification_df['Flag Value'], classification_df['Solar-Base']))
lucc_cf = dict(zip(classification_df['Flag Value'], classification_df['CF']))
lucc_cf_noWater = dict(zip(classification_df['Flag Value'], classification_df['CF_noWater']))
slope_1km = slope_1km.rename({'y': 'latitude', 'x': 'longitude'})

In [ ]:
# 定义函数，将土地利用类型的值映射到对应的权重
def lucc_wind_process_weighted(x):
    return lucc_Wind_weights.get(x, np.nan)  # 如果找不到对应的值，则返回 NaN

def lucc_solar_process_weighted(x):
    return lucc_Solar_weights.get(x, np.nan)  # 如果找不到对应的值，则返回 NaN

def lucc_cf_noWater_weighted(x):
    return lucc_cf_noWater.get(x, np.nan)  # 如果找不到对应的值，则返回 NaN

def lucc_cf_weighted(x):
    return lucc_cf.get(x, np.nan)  # 如果找不到对应的值，则返回 NaN
# 使用 map_blocks 将土地利用类型数据按块映射为对应权重

lucc_Wind_025d = lucc_025d.map_blocks(np.vectorize(lucc_wind_process_weighted))
lucc_Solar_025d = lucc_025d.map_blocks(np.vectorize(lucc_solar_process_weighted))
lucc_cf_noWater_025d = lucc_025d.map_blocks(np.vectorize(lucc_cf_noWater_weighted))
lucc_cf_025d = lucc_025d.map_blocks(np.vectorize(lucc_cf_weighted))
lucc_Wind_1km = lucc_1km.map_blocks(np.vectorize(lucc_wind_process_weighted))
lucc_Solar_1km = lucc_1km.map_blocks(np.vectorize(lucc_solar_process_weighted))
lucc_cf_noWater_1km = lucc_1km.map_blocks(np.vectorize(lucc_cf_noWater_weighted))
ucc_cf_1km = lucc_1km.map_blocks(np.vectorize(lucc_cf_weighted))


def Wind_slope_process(x):
    conditions = [(x >= 0) | (x <= 3), (x > 3)] #as Solar
    #conditions = [(x >= 0) | (x <= 15), (x > 15)] #Conservative
    #conditions = [(x >= 0) | (x <= 20), (x > 20)] #Base
    #conditions = [(x >= 0) | (x <= 25), (x > 25)] #Open
    slopes = [1, 0]
    return np.select(conditions, slopes, default=np.nan)
wind_slope_025d = Wind_slope_process(slope_025d)
wind_slope_1km = Wind_slope_process(slope_1km)

def Solar_slope_process(x):
    #conditions = [(x >= 0) | (x <= 3), (x > 3)] #Conservative
    conditions = [(x >= 0) | (x <= 5), (x > 5)] #Base
    #conditions = [(x >= 0) | (x <= 7), (x > 7)] #Open
    slopes = [1, 0]
    return np.select(conditions, slopes, default=np.nan)
solar_slope_025d = Solar_slope_process(slope_025d)
solar_slope_1km = Solar_slope_process(slope_1km)

def elevation_process(x):
    conditions = [x<=3000, x>3000]
    elevations = [1, 0]
    return np.select(conditions, elevations, default=np.nan)
elevation_025d = elevation_process(elevation_025d)
elevation_1km = elevation_process(elevation_1km)

def protected_process(x):
    conditions = [x==0, x>0]
    protected = [1, 0]
    return np.select(conditions, protected, default=np.nan)
protected_025d = protected_process(protected_025d)
protected_1km = protected_process(protected_1km)

def CF_process_wind(x):
    conditions = [(x < 3) | (x > 25), (x < 5), (x < 12), (x <= 25)]
    CFs = [0, 125*(x+1e-7-3)/2500, (250 + 2250/7 * (x-5))/2500, 1]
    return np.select(conditions, CFs, default=np.nan)

def pv_power(T, S, wind10):  # 光伏小时容量因子计算函数
    # T: 环境温度 (摄氏度)
    # S: 太阳辐射量 (单位：W/m²)
    # wind10: 10米高度的风速 (单位：m/s)
    
    # 计算光伏组件温度 (T_p)
    # T_p 表示组件的实际温度，它受环境温度、太阳辐射量以及风速的影响
    T_p = 4.3 + 0.943 * T + 0.028 * S / 3600 - 1.528 * wind10

    # 计算温度对光伏组件效率的影响 (TEM)
    # 组件温度高于25度会降低效率，每升高1度，效率降低0.5%
    TEM = (1 - 0.005 * (T_p - 25))

    # 系统效率 (SYS)
    # 包括逆变器、线缆等系统损耗，假设为85%
    SYS = 0.85

    # 返回光伏小时容量因子
    return TEM * SYS


def pv_power1(T, S, wind10):  # 光伏小时容量因子计算函数（改进版）
    # T: 环境温度 (摄氏度)
    # S: 太阳辐射量 (单位：W/m²)
    # wind100: 10米高度的风速 (单位：m/s)

    # 计算光伏组件温度 (T_p)
    # 与前一个函数相同，计算组件温度
    T_p = 4.3 + 0.943 * T + 0.028 * S / 3600 - 1.528 * wind10

    # 计算温度对光伏组件效率的影响 (TEM)
    TEM = (1 - 0.005 * (T_p - 25))

    # 系统效率 (SYS)
    SYS = 0.85

    # 返回光伏小时容量因子
    # 在这里，计算的单位从 W 转换为 kWh，S/1000 将 S 单位从 W/m² 转换为 kW/m²，除以 3600 以获得小时数的因子
    return S / 1000 / 3600 * TEM * SYS



In [ ]:
for i in range(0, 1440, 60):
    print(i)
    data = xr.open_dataset(rf"D:\Workspace\average_2023_segment_{i}.nc", engine='netcdf4')
    CF_wind = data["CF_wind_avg"][0:8760, :, :]
    
    slope_1km_chunk = slope_1km[:, i*30:i*30+1800]
    
    lucc_cf_noWater_chunk = lucc_cf_noWater_025d[:, i:i+60]
    lucc_cf_chunk = lucc_cf_025d[:, i:i+60]
    
    lucc_Wind_1km_chunk = lucc_Wind_1km[:, i*30:i*30+1800]
    lucc_Wind_025d_chunk = lucc_Wind_025d[:, i:i+60]
    lucc_Solar_1km_chunk = lucc_Solar_1km[:, i*30:i*30+1800]
    lucc_Solar_025d_chunk = lucc_Solar_025d[:, i:i+60]
    
    wind_slope_025d_chunk = wind_slope_025d[:, i:i+60]
    solar_slope_025d_chunk = solar_slope_025d[:, i:i+60]
    wind_slope_1km_chunk = wind_slope_1km[:, i*30:i*30+1800]
    solar_slope_1km_chunk = solar_slope_1km[:, i*30:i*30+1800]
    
    elevation_025d_chunk = elevation_025d[:, i:i+60]
    protected_025d_chunk = protected_025d[ :, i:i+60]
    elevation_1km_chunk = elevation_1km[:, i*30:i*30+1800]
    protected_1km_chunk = protected_1km[:, i*30:i*30+1800]
    
    
    
    Cover_noWater = lucc_cf_noWater_chunk
    pv_cover_chunk = lucc_cf_chunk * solar_slope_025d_chunk * elevation_025d_chunk * protected_025d_chunk
    wind_cover_chunk = lucc_cf_chunk * wind_slope_025d_chunk * elevation_025d_chunk * protected_025d_chunk
    
    pv_base_025d_chunk = lucc_Solar_025d_chunk * solar_slope_025d_chunk * elevation_025d_chunk * protected_025d_chunk
    wind_base_025d_chunk = lucc_Wind_025d_chunk* wind_slope_025d_chunk * elevation_025d_chunk * protected_025d_chunk
    
    pv_base_1km_chunk = lucc_Solar_1km_chunk * solar_slope_1km_chunk * elevation_1km_chunk * protected_1km_chunk
    wind_base_1km_chunk = lucc_Wind_1km_chunk* wind_slope_1km_chunk * elevation_1km_chunk * protected_1km_chunk
    
    # 将 CP_wind 转换为 xarray DataArray
    Cover_noWater_da = xr.DataArray(Cover_noWater, dims=[ "latitude", "longitude"], 
                          coords={ "latitude": CF_wind.latitude, 
                                  "longitude": CF_wind.longitude},
                              name=f"Cover_noWater")
    
    pv_cover_chunk_da = xr.DataArray(pv_cover_chunk, dims=[ "latitude", "longitude"], 
                      coords={ "latitude": CF_wind.latitude, 
                              "longitude": CF_wind.longitude},
                          name=f"Cover_pv")

    wind_cover_chunk_da = xr.DataArray(wind_cover_chunk, dims=[ "latitude", "longitude"], 
                  coords={ "latitude": CF_wind.latitude, 
                          "longitude": CF_wind.longitude},
                      name=f"Cover_wind")
    
    pv_base_025d_chunk_da = xr.DataArray(pv_base_025d_chunk, dims=[ "latitude", "longitude"], 
                      coords={ "latitude": CF_wind.latitude, 
                              "longitude": CF_wind.longitude},
                          name=f"pv_base_025d")

    wind_base_025d_chunk_da = xr.DataArray(wind_base_025d_chunk, dims=[ "latitude", "longitude"], 
                  coords={ "latitude": CF_wind.latitude, 
                          "longitude": CF_wind.longitude},
                      name=f"wind_base_025d")
    
    pv_base_1km_chunk_da = xr.DataArray(pv_base_1km_chunk, dims=[ "latitude", "longitude"], 
                      coords={ "latitude": slope_1km_chunk.latitude, 
                              "longitude": slope_1km_chunk.longitude},
                          name=f"pv_base_1km")

    wind_base_1km_chunk_da = xr.DataArray(wind_base_1km_chunk, dims=[ "latitude", "longitude"], 
                  coords={ "latitude": slope_1km_chunk.latitude, 
                          "longitude": slope_1km_chunk.longitude},
                      name=f"wind_base_1km")
    
    Cover_noWater_da.to_netcdf(rf"E:\Workspace\Data_temp\Cover_noWater_segment_{i}.nc")
    pv_cover_chunk_da.to_netcdf(rf"E:\Workspace\Data_temp\pv_cover_chunk_{i}.nc")
    wind_cover_chunk_da.to_netcdf(rf"E:\Workspace\Data_temp\wind_cover_chunk_{i}.nc")
    pv_base_025d_chunk_da.to_netcdf(rf"E:\Workspace\Data_temp\pv_base_025d_chunk_{i}.nc")    
    wind_base_025d_chunk_da.to_netcdf(rf"E:\Workspace\Data_temp\wind_base_025d_chunk_{i}.nc")
    pv_base_1km_chunk_da.to_netcdf(rf"E:\Workspace\Data_temp\pv_base_1km_chunk_{i}.nc")    
    wind_base_1km_chunk_da.to_netcdf(rf"E:\Workspace\Data_temp\wind_base_1km_chunk_{i}.nc")

In [ ]:
import os
import re
import xarray as xr
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ========================================
# 1. 设置路径与文件读取
# ========================================
folder = "E:/Workspace/Data_temp"  # 修改为你的路径
tif_path = "F:/Workspace/Snow/2000-2020_025d_p50_mean_merge_new.tif"
pattern = re.compile(r"wind_cover_chunk_(\d+).nc")

# 获取并排序文件
files = sorted(
    [f for f in os.listdir(folder) if pattern.search(f)],
    key=lambda x: int(pattern.search(x).group(1))
)

# 打开并按经度拼接
datasets = [xr.open_dataset(os.path.join(folder, f)) for f in files]
combined = xr.concat(datasets, dim="longitude")

for ds in datasets:
    ds.close()

# ========================================
# 2. 提取 Cover_wind 变量
# ========================================
var = 'Cover_wind'
if var not in combined:
    raise KeyError(f"Variable '{var}' not found in dataset. Found: {list(combined.data_vars)}")

wind_cover = combined[var]
lon = wind_cover['longitude']
lat = wind_cover['latitude']

# ========================================
# 3. 读取掩膜 tif 并转换为 0/1 掩码
# ========================================
with rasterio.open(tif_path) as src:
    mask_data = src.read(1)
    mask_transform = src.transform
    mask_bounds = src.bounds

# 创建 0/1 掩膜：小于 50 设为 1，其余为 0
binary_mask = np.where(mask_data < 50, 1, 0)
binary_mask = np.roll(binary_mask, shift=720, axis=1)

# 检查维度是否匹配
if binary_mask.shape != wind_cover.shape[-2:]:
    raise ValueError(f"掩膜形状 {binary_mask.shape} 与 wind_cover {wind_cover.shape} 不匹配，请检查分辨率对齐。")

# ========================================
# 4. 应用掩膜与 wind_cover 相乘
# ========================================
masked_wind_cover = wind_cover * binary_mask

# ========================================
# 5. 绘图（可选）
# ========================================
fig = plt.figure(figsize=(32, 16))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.LAND, facecolor='lightgray')

p = ax.pcolormesh(lon, lat, masked_wind_cover.squeeze(), cmap='viridis', vmin=0, vmax=1)
cbar = plt.colorbar(p, ax=ax, orientation='vertical', shrink=0.7)
cbar.set_label("Wind Cover (masked by <50 Snow Cover)")
plt.title("Masked Global Wind Cover (<50 Snow Cover Areas Only)")
plt.tight_layout()
plt.show()

# ========================================
# 6. 保存结果为新的 nc 文件
# ========================================
combined["Cover_wind_masked"] = masked_wind_cover
combined.to_netcdf(os.path.join(folder, "wind_cover_global_masked.nc"))
print("Masked dataset saved.")


In [ ]:
import os
import re
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ========================================
# 1. 设置路径与文件读取
# ========================================
folder = "E:/Workspace/Data_temp"  # 修改为你的路径
pattern = re.compile(r"pv_cover_chunk_(\d+).nc")

# 获取并排序文件
files = sorted(
    [f for f in os.listdir(folder) if pattern.search(f)],
    key=lambda x: int(pattern.search(x).group(1))
)

# 打开并按经度拼接
datasets = [xr.open_dataset(os.path.join(folder, f)) for f in files]
combined = xr.concat(datasets, dim="longitude")

# 可选：关闭原始文件资源
for ds in datasets:
    ds.close()

# ========================================
# 2. 提取 wind_cover 变量并绘图
# ========================================
var = 'Cover_pv'
if var not in combined:
    raise KeyError(f"Variable '{var}' not found in dataset. Found: {list(combined.data_vars)}")

wind_cover = combined[var]

# 获取经纬度
lon = wind_cover['longitude']
lat = wind_cover['latitude']

# ========================================
# 3. 可视化带国界的地图
# ========================================
fig = plt.figure(figsize=(16, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.LAND, facecolor='lightgray')

# 绘图
p = ax.pcolormesh(lon, lat, wind_cover.squeeze(), cmap='viridis', vmin=0, vmax=1)
cbar = plt.colorbar(p, ax=ax, orientation='vertical', shrink=0.7)
cbar.set_label("PV Cover")

plt.title("Global PV Cover (Merged by Longitude Chunks)")
plt.tight_layout()
plt.show()

# ========================================
# 4. 可选保存合并结果
# ========================================
combined.to_netcdf(os.path.join(folder, "pv_cover_global_merged.nc"))
print("Merged dataset saved.")


In [ ]:
import os
import glob
import numpy as np
import rasterio

# 设置数据路径和输出路径
data_dir = "F:/Workspace/Snow"  # ← 修改为你的.tif文件所在路径
output_file = "2000-2020_1km_p50_mean.tif"

# 匹配文件名格式
pattern = "clm_snow.cover_esa.modis_p50_1km_s0..0cm_*.tif"
tif_files = sorted(glob.glob(os.path.join(data_dir, pattern)))

# 初始化变量
sum_array = None
count_array = None
profile = None

for idx, tif_path in enumerate(tif_files):
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(np.float32)
        nodata = src.nodata
        if profile is None:
            profile = src.profile
            sum_array = np.zeros(data.shape, dtype=np.float64)
            count_array = np.zeros(data.shape, dtype=np.uint16)

        # 处理 nodata（假如为 None，则跳过）
        if nodata is not None:
            data[data == nodata] = np.nan

        # 有效像素掩膜
        valid_mask = ~np.isnan(data)
        sum_array[valid_mask] += data[valid_mask]
        count_array[valid_mask] += 1

        print(f"[{idx+1}/{len(tif_files)}] 已处理: {os.path.basename(tif_path)}")

# 计算均值（避免除以0）
mean_array = np.divide(sum_array, count_array, out=np.full_like(sum_array, np.nan), where=count_array != 0)

# 更新 profile 信息
profile.update(dtype=rasterio.float32, count=1, compress="deflate", nodata=np.nan)

# 写出结果
with rasterio.open(output_file, "w", **profile) as dst:
    dst.write(mean_array.astype(np.float32), 1)

print(f"\n✅ 均值图像已保存到：{output_file}")


In [ ]:
wind_cover.close()
Cover_noWater.close()

In [ ]:
wind_cover

In [ ]:
plt.imshow(lucc_cf_noWater_025d)
plt.colorbar()

In [ ]:
plt.imshow(elevation_025d_ori)
plt.colorbar()

In [ ]:
elevation_025d_ori = xr.open_dataset(r"F:\Workspace\DATA_DEM\merged_global_map_025d.tif")["band_data"][0, :, :]

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
for longitude_idx in range(7,8):  # Assuming data is split by longitude into 24 chunks
    print(f"Processing chunk {longitude_idx + 1}")
    # Load wind and solar data for the current chunk
    data = xr.open_dataset(f"D:/Workspace/average_2023_segment_{longitude_idx * 60}.nc")

    wind_output = np.array(data["CF_wind_avg"][:, 195, 20])  # Wind power output for current grid point
    pv_output = np.array(data["CF_pv_avg"][:, 195,20])  # PV power output for current grid point
    output_data = np.append(wind_output,pv_output)
    output_df = pd.DataFrame(output_data)
    output_df.to_csv(f"CF_NM_2023.csv", index=False)

In [ ]:
np.shape(wind_output)

In [ ]:
np.shape(output_data)

In [ ]:
output_df

In [ ]:
import xarray as xr
data = xr.open_dataset(f"D:/Workspace/average_2023_segment_{0 * 60}.nc")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm
plt.imshow(np.array(data.to_array())[0,0,:,:])